# Fine-tune VietOCR VGG19-bn Transformer on HocBa cells

Notebook này chạy **trực tiếp trong cell**, không ghi code ra `.py`, không gọi script train qua process ngoài.

Bạn chỉ cần gắn Kaggle Dataset chứa folder `ocr_data` đã chuẩn bị sẵn, gồm `annotation_train.txt`, `annotation_val.txt`, `annotation_test.txt` và `images/`.


## Bắt Buộc Restart Kernel

Nếu cell install báo kernel đang giữ `numpy/torch/torchvision` cũ trong RAM, không có cách sửa bằng chạy lại cell. Trên Kaggle phải vào **Session options -> Restart session**, sau đó chạy từ đầu.

Phiên bản đúng sau cell install phải là `numpy 1.26.x`, `torch 2.4.1+cu118`, `torchvision 0.19.1+cu118`.


## Setup Paths

In [ ]:
from __future__ import annotations

from pathlib import Path
from types import SimpleNamespace
from typing import Any, Dict, List, Optional, Sequence, Set, Tuple
from collections import Counter

import csv
import json
import math
import os
import random
import shutil
import time

WORK_ROOT = Path('/kaggle/working/hocba_vietocr_run') if Path('/kaggle').exists() else Path.cwd() / 'hocba_vietocr_run'
INPUT_ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd()
OCR_DATA_OVERRIDE = None
# OCR_DATA_OVERRIDE = Path('/home/namhoai/WorkSpace/Hoctap/CS338/hocba_vietocr_fit/ocr_data')

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print('WORK_ROOT:', WORK_ROOT)
print('INPUT_ROOT:', INPUT_ROOT)


import os
import sys
import subprocess
import importlib.metadata as metadata

TARGETS = {
    'numpy': '1.26.',
    'torch': '2.4.1',
    'torchvision': '0.19.1',
}


def dist_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


def loaded_version(module_name):
    module = sys.modules.get(module_name)
    return getattr(module, '__version__', None) if module is not None else None

installed_before = {name: dist_version(name) for name in TARGETS}
loaded_before = {name: loaded_version(name) for name in TARGETS}
print('installed before:', installed_before)
print('loaded before:', loaded_before)

loaded_wrong = any(
    version is not None and not version.startswith(prefix)
    for name, prefix in TARGETS.items()
    for version in [loaded_before[name]]
)
installed_wrong = any(
    installed_before[name] is None or not installed_before[name].startswith(prefix)
    for name, prefix in TARGETS.items()
)

if loaded_wrong or installed_wrong:
    print('Cài lại dependency đúng cho Kaggle P100...')
    !python -m pip uninstall -y -q torch torchvision torchaudio numpy opencv-python opencv-python-headless opencv-contrib-python imgaug albumentations vietocr
    !python -m pip install -q --no-cache-dir numpy==1.26.4
    !python -m pip install -q --no-cache-dir opencv-python==4.10.0.84 opencv-python-headless==4.10.0.84 imgaug==0.4.0 albumentations==1.4.2 scikit-image==0.23.2 Pillow==10.2.0 lmdb==1.4.1 einops==0.2.0 gdown==4.4.0 PyYAML==6.0.2 tqdm==4.66.5 wandb==0.19.8 prefetch-generator==1.0.1
    !python -m pip install -q --no-cache-dir --no-deps vietocr==0.3.13
    !python -m pip install -q --upgrade --force-reinstall --no-cache-dir torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1 --index-url https://download.pytorch.org/whl/cu118
    !python -m pip install -q --force-reinstall --no-cache-dir --no-deps numpy==1.26.4

    print('installed after:', {name: dist_version(name) for name in TARGETS})
    print('Dependency đã cài xong. Kernel sẽ tự restart để xóa module cũ khỏi RAM.')
    print('Sau khi kernel restart, chạy lại notebook từ đầu. Nếu dùng Run All, bấm Run All thêm một lần nữa.')
    os._exit(0)

import cv2
import numpy as np
import torch
import torchvision
import yaml
from PIL import Image, ImageDraw, ImageFont, ImageOps

import vietocr
print('numpy:', np.__version__)
print('cv2:', cv2.__version__)
print('torch:', torch.__version__, 'cuda:', torch.version.cuda)
print('torchvision:', torchvision.__version__)
print('vietocr module:', vietocr.__file__)
assert np.__version__.startswith('1.26.'), f'NumPy phải là 1.26.x cho imgaug, hiện tại là {np.__version__}'
assert torch.__version__.startswith('2.4.1'), f'Torch phải là 2.4.1 cho Kaggle P100, hiện tại là {torch.__version__}'
assert torchvision.__version__.startswith('0.19.1'), f'Torchvision phải là 0.19.1, hiện tại là {torchvision.__version__}'
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0), 'capability:', torch.cuda.get_device_capability(0))
    print('compiled archs:', torch.cuda.get_arch_list())


In [ ]:
import sys
_loaded = sorted(
    m for m in sys.modules
    if m == 'torch' or m.startswith('torch.')
    or m == 'torchvision' or m.startswith('torchvision.')
    or m == 'numpy' or m.startswith('numpy.')
    or m == 'imgaug' or m.startswith('imgaug.')
)
if _loaded:
    print('Modules đã loaded trong kernel hiện tại:', _loaded[:30], '...' if len(_loaded) > 30 else '')
    try:
        import numpy as _np
        import torch as _torch
        import torchvision as _torchvision
        _ok = (
            _np.__version__.startswith('1.26.')
            and _torch.__version__.startswith('2.4.1')
            and _torchvision.__version__.startswith('0.19.1')
        )
        print('current numpy/torch/torchvision:', _np.__version__, _torch.__version__, _torchvision.__version__)
    except Exception as _e:
        _ok = False
        print('Không đọc được version hiện tại:', repr(_e))
    if not _ok:
        raise RuntimeError('Kernel đang giữ dependency cũ trong RAM. Vào Kaggle: Session options -> Restart session, rồi Run all từ đầu. Không chạy tiếp cell train trong kernel này.')
    print('Dependency trong RAM đã đúng version, bỏ qua cài lại.')
else:
    !python -m pip uninstall -y -q torch torchvision torchaudio numpy opencv-python opencv-python-headless opencv-contrib-python imgaug albumentations vietocr
    !python -m pip install -q --no-cache-dir numpy==1.26.4
    !python -m pip install -q --no-cache-dir opencv-python==4.10.0.84 opencv-python-headless==4.10.0.84 imgaug==0.4.0 albumentations==1.4.2 scikit-image==0.23.2 Pillow==10.2.0 lmdb==1.4.1 einops==0.2.0 gdown==4.4.0 PyYAML==6.0.2 tqdm==4.66.5 wandb==0.19.8 prefetch-generator==1.0.1
    !python -m pip install -q --no-cache-dir --no-deps vietocr==0.3.13
    !python -m pip install -q --upgrade --force-reinstall --no-cache-dir torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1 --index-url https://download.pytorch.org/whl/cu118
    # torch/torchvision có thể kéo numpy mới lên lại; pin lần cuối cùng trước khi import Python modules.
    !python -m pip install -q --force-reinstall --no-cache-dir --no-deps numpy==1.26.4

import cv2
import numpy as np
import torch
import torchvision
import yaml
from PIL import Image, ImageDraw, ImageFont, ImageOps

import vietocr
print('numpy:', np.__version__)
print('cv2:', cv2.__version__)
print('torch:', torch.__version__, 'cuda:', torch.version.cuda)
print('torchvision:', torchvision.__version__)
print('vietocr module:', vietocr.__file__)
assert np.__version__.startswith('1.26.'), f'NumPy phải là 1.26.x cho imgaug, hiện tại là {np.__version__}'
assert torch.__version__.startswith('2.4.1'), f'Torch phải là 2.4.1 cho Kaggle P100, hiện tại là {torch.__version__}'
assert torchvision.__version__.startswith('0.19.1'), f'Torchvision phải là 0.19.1, hiện tại là {torchvision.__version__}'
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0), 'capability:', torch.cuda.get_device_capability(0))
    print('compiled archs:', torch.cuda.get_arch_list())


## Load OCR Dataset

In [ ]:
def looks_like_ocr_data(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / 'annotation_train.txt').exists()
        and (path / 'annotation_val.txt').exists()
        and (path / 'images').exists()
    )


def find_ocr_data() -> Path:
    if OCR_DATA_OVERRIDE is not None:
        p = Path(OCR_DATA_OVERRIDE)
        if looks_like_ocr_data(p):
            return p
        raise FileNotFoundError(f'OCR_DATA_OVERRIDE không phải ocr_data hợp lệ: {p}')

    candidates = []
    if looks_like_ocr_data(INPUT_ROOT):
        candidates.append(INPUT_ROOT)
    candidates.extend(p for p in INPUT_ROOT.rglob('ocr_data') if looks_like_ocr_data(p))
    candidates.extend(p.parent for p in INPUT_ROOT.rglob('annotation_train.txt') if looks_like_ocr_data(p.parent))

    unique = []
    seen = set()
    for p in candidates:
        rp = p.resolve()
        if rp not in seen:
            unique.append(p)
            seen.add(rp)

    if not unique:
        raise FileNotFoundError(
            'Không tìm thấy folder ocr_data trong /kaggle/input. '
            'Dataset cần có annotation_train.txt, annotation_val.txt và images/.'
        )
    if len(unique) > 1:
        print('Tìm thấy nhiều ocr_data, dùng cái đầu tiên:')
        for p in unique:
            print(' -', p)
    return unique[0]

INPUT_OCR_DATA = find_ocr_data()
OCR_DATA = WORK_ROOT / 'ocr_data'
if OCR_DATA.exists():
    shutil.rmtree(OCR_DATA)
shutil.copytree(INPUT_OCR_DATA, OCR_DATA)

print('copied ocr_data:', INPUT_OCR_DATA, '->', OCR_DATA)
!find {OCR_DATA} -maxdepth 2 -type f | head -20
!wc -l {OCR_DATA/'annotation_train.txt'} {OCR_DATA/'annotation_val.txt'} {OCR_DATA/'annotation_test.txt'}

summary_path = OCR_DATA / 'summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(json.dumps(summary, ensure_ascii=False, indent=2)[:3000])


## Audit Helpers

In [ ]:
def read_annotation(path: Path) -> List[Tuple[str, str, str]]:
    split = path.stem.replace('annotation_', '')
    rows: List[Tuple[str, str, str]] = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line:
                continue
            if '\t' not in line:
                raise ValueError(f'Malformed annotation line without tab: {path}:{len(rows) + 1}: {line!r}')
            image_relpath, label = line.split('\t', 1)
            rows.append((split, image_relpath, label))
    return rows


def load_rows(data_root: Path) -> List[Tuple[str, str, str]]:
    rows: List[Tuple[str, str, str]] = []
    for split in ('train', 'val', 'test'):
        annotation_path = data_root / f'annotation_{split}.txt'
        if annotation_path.exists():
            rows.extend(read_annotation(annotation_path))
    return rows


def percentile(values: Sequence[float], pct: float) -> float:
    if not values:
        return 0.0
    return float(np.percentile(np.array(values, dtype=np.float32), pct))


def tenengrad(gray: np.ndarray) -> float:
    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    return float(np.mean(sobel_x * sobel_x + sobel_y * sobel_y))


def image_metrics(path: Path) -> Dict[str, Any]:
    image = ImageOps.exif_transpose(Image.open(path)).convert('RGB')
    arr = np.asarray(image)
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    h, w = gray.shape[:2]
    lap_var = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    gaussian = cv2.GaussianBlur(gray, (3, 3), 0)
    return {
        'width': w,
        'height': h,
        'aspect': float(w / h) if h else 0.0,
        'brightness': float(gray.mean()),
        'contrast': float(gray.std()),
        'laplacian_var': lap_var,
        'laplacian_var_after_gaussian': float(cv2.Laplacian(gaussian, cv2.CV_64F).var()),
        'tenengrad': tenengrad(gray),
    }


def reasons_for(record: Dict[str, Any], thresholds: Dict[str, float], args: SimpleNamespace) -> List[str]:
    reasons: List[str] = []
    if record['laplacian_var'] < thresholds['blur']:
        reasons.append('low_laplacian_blur_score')
    if record['contrast'] < thresholds['contrast']:
        reasons.append('low_contrast')
    if record['brightness'] < args.brightness_low:
        reasons.append('too_dark')
    if record['brightness'] > args.brightness_high:
        reasons.append('too_bright')
    if record['width'] < args.min_width:
        reasons.append('too_narrow')
    if record['height'] < args.min_height:
        reasons.append('too_short')
    return reasons


def shrink_to_box(image: Image.Image, box: Tuple[int, int]) -> Image.Image:
    max_w, max_h = box
    scale = min(max_w / image.width, max_h / image.height, 1.0)
    new_size = (max(1, int(image.width * scale)), max(1, int(image.height * scale)))
    return image.resize(new_size, Image.Resampling.LANCZOS)


def create_contact_sheet(records: Sequence[Dict[str, Any]], data_root: Path, output_path: Path, title: str, cols: int = 5, cell_size: Tuple[int, int] = (220, 130)) -> None:
    if not records:
        return
    rows = math.ceil(len(records) / cols)
    header_h = 34
    label_h = 48
    sheet = Image.new('RGB', (cols * cell_size[0], header_h + rows * cell_size[1]), 'white')
    draw = ImageDraw.Draw(sheet)
    font = ImageFont.load_default()
    draw.text((8, 8), title, fill=(0, 0, 0), font=font)
    for idx, record in enumerate(records):
        col = idx % cols
        row = idx // cols
        x = col * cell_size[0]
        y = header_h + row * cell_size[1]
        draw.rectangle((x, y, x + cell_size[0] - 1, y + cell_size[1] - 1), outline=(210, 210, 210))
        image = Image.open(data_root / record['image']).convert('RGB')
        thumb = shrink_to_box(image, (cell_size[0] - 12, cell_size[1] - label_h - 10))
        sheet.paste(thumb, (x + (cell_size[0] - thumb.width) // 2, y + 6))
        text = record['label']
        if len(text) > 34:
            text = text[:31] + '...'
        meta = f"{record['split']} lap={record['laplacian_var']:.1f} c={record['contrast']:.1f}"
        draw.text((x + 6, y + cell_size[1] - 38), text, fill=(0, 0, 0), font=font)
        draw.text((x + 6, y + cell_size[1] - 20), meta, fill=(80, 80, 80), font=font)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    sheet.save(output_path, quality=92)


def write_csv(path: Path, records: Sequence[Dict[str, Any]]) -> None:
    fields = ['split', 'image', 'label', 'width', 'height', 'brightness', 'contrast', 'laplacian_var', 'laplacian_var_after_gaussian', 'tenengrad', 'reasons']
    with path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for record in records:
            writer.writerow({field: record.get(field, '') for field in fields})


def audit_cell_quality(data_root: Path, output: Path, sample_size: int = 300, seed: int = 338, copy_flagged: int = 200) -> Dict[str, Any]:
    args = SimpleNamespace(
        blur_percentile=2.0,
        contrast_percentile=1.0,
        brightness_low=25.0,
        brightness_high=245.0,
        min_width=10,
        min_height=10,
    )
    if output.exists():
        shutil.rmtree(output)
    (output / 'flagged_samples').mkdir(parents=True, exist_ok=True)
    rows = load_rows(data_root)
    records = []
    for split, image_relpath, label in rows:
        records.append({'split': split, 'image': image_relpath, 'label': label, **image_metrics(data_root / image_relpath)})
    thresholds = {
        'blur': percentile([r['laplacian_var'] for r in records], args.blur_percentile),
        'contrast': percentile([r['contrast'] for r in records], args.contrast_percentile),
    }
    flagged = []
    reason_counts = Counter()
    for record in records:
        reasons = reasons_for(record, thresholds, args)
        record['reasons'] = '|'.join(reasons)
        if reasons:
            flagged.append(record)
            reason_counts.update(reasons)
    write_csv(output / 'quality_all.csv', records)
    write_csv(output / 'quality_flagged.csv', flagged)
    random.seed(seed)
    sample_records = random.sample(records, min(sample_size, len(records)))
    create_contact_sheet(sample_records, data_root, output / 'contact_sheet_random_300.jpg', f'Random review sample: {len(sample_records)} cells')
    worst_blur = sorted(records, key=lambda r: r['laplacian_var'])[: min(sample_size, len(records))]
    create_contact_sheet(worst_blur, data_root, output / 'contact_sheet_worst_blur_300.jpg', f'Worst blur-score sample: {len(worst_blur)} cells')
    if copy_flagged:
        for idx, record in enumerate(sorted(flagged, key=lambda r: (r['laplacian_var'], r['contrast']))[:copy_flagged]):
            src = data_root / record['image']
            dst = output / 'flagged_samples' / f"{idx:04d}_{record['split']}_{record['laplacian_var']:.1f}_{record['contrast']:.1f}{Path(record['image']).suffix}"
            shutil.copy2(src, dst)
    summary = {
        'data_root': str(data_root),
        'total_images': len(records),
        'flagged_images': len(flagged),
        'thresholds': thresholds,
        'reason_counts': dict(reason_counts),
        'by_split': dict(Counter(r['split'] for r in records)),
        'flagged_by_split': dict(Counter(r['split'] for r in flagged)),
    }
    with (output / 'summary.json').open('w', encoding='utf-8') as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
        f.write('\n')
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    return summary


## Run Audit

In [ ]:
QUALITY_OUT = WORK_ROOT / 'quality_audit'
audit_summary = audit_cell_quality(OCR_DATA, QUALITY_OUT, sample_size=300)
print('Random sheet:', QUALITY_OUT / 'contact_sheet_random_300.jpg')
print('Worst blur sheet:', QUALITY_OUT / 'contact_sheet_worst_blur_300.jpg')


In [ ]:
from IPython.display import display

random_sheet = QUALITY_OUT / 'contact_sheet_random_300.jpg'
worst_sheet = QUALITY_OUT / 'contact_sheet_worst_blur_300.jpg'
if random_sheet.exists():
    display(Image.open(random_sheet))
if worst_sheet.exists():
    display(Image.open(worst_sheet))
print('Flagged CSV:', QUALITY_OUT / 'quality_flagged.csv')


## Optional: Exclude Bad Images

In [ ]:
def load_exclude_list(path: Path) -> Set[str]:
    excluded: Set[str] = set()
    if not path.exists():
        return excluded
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#'):
                excluded.add(line)
    return excluded


def filter_annotation(input_path: Path, output_path: Path, excluded: Set[str]) -> Dict[str, int]:
    kept = 0
    removed = 0
    with input_path.open('r', encoding='utf-8') as src, output_path.open('w', encoding='utf-8') as dst:
        for line in src:
            image_relpath = line.rstrip('\n').split('\t', 1)[0]
            if image_relpath in excluded:
                removed += 1
                continue
            dst.write(line)
            kept += 1
    return {'kept': kept, 'removed': removed}


EXCLUDE_IMAGES = [
    # 'images/train/hoctap/xxx_cell000.jpg',
]
excluded = set(EXCLUDE_IMAGES)
if not excluded:
    print('No images selected for exclusion. Filtered annotations will match originals.')

for split in ('train', 'val', 'test'):
    stats = filter_annotation(OCR_DATA / f'annotation_{split}.txt', OCR_DATA / f'annotation_{split}_filtered.txt', excluded)
    print(split, stats)

TRAIN_ANNOTATION = 'annotation_train_filtered.txt'
VALID_ANNOTATION = 'annotation_val_filtered.txt'
TEST_ANNOTATION = 'annotation_test_filtered.txt'
!wc -l {OCR_DATA/TRAIN_ANNOTATION} {OCR_DATA/VALID_ANNOTATION} {OCR_DATA/TEST_ANNOTATION}


## Config

In [ ]:
BASE_CONFIG_YAML = r'''# Full config: base VietOCR + VGG19-bn Transformer pretrained model.
vocab: 'aAàÀảẢãÃáÁạẠăĂằẰẳẲẵẴắẮặẶâÂầẦẩẨẫẪấẤậẬbBcCdDđĐeEèÈẻẺẽẼéÉẹẸêÊềỀểỂễỄếẾệỆfFgGhHiIìÌỉỈĩĨíÍịỊjJkKlLmMnNoOòÒỏỎõÕóÓọỌôÔồỒổỔỗỖốỐộỘơƠờỜởỞỡỠớỚợỢpPqQrRsStTuUùÙủỦũŨúÚụỤưƯừỪửỬữỮứỨựỰvVwWxXyYỳỲỷỶỹỸýÝỵỴzZ0123456789!"#$%&''()*+,-./:;<=>?@[\]^_`{|}~ '

device: cuda:0

pretrain: https://vocr.vn/data/vietocr/vgg_transformer.pth
weights: ./hocba_vietocr_fit/weights/vgg19_transformer_hocba.pth

backbone: vgg19_bn
seq_modeling: transformer

cnn:
  pretrained: true
  ss:
    - [2, 2]
    - [2, 2]
    - [2, 1]
    - [2, 1]
    - [1, 1]
  ks:
    - [2, 2]
    - [2, 2]
    - [2, 1]
    - [2, 1]
    - [1, 1]
  hidden: 256

transformer:
  d_model: 256
  nhead: 8
  num_encoder_layers: 6
  num_decoder_layers: 6
  dim_feedforward: 2048
  max_seq_length: 1024
  pos_dropout: 0.1
  trans_dropout: 0.1

optimizer:
  max_lr: 0.0001
  pct_start: 0.15

trainer:
  batch_size: 32
  print_every: 50
  valid_every: 250
  iters: 6000
  export: ./hocba_vietocr_fit/weights/vgg19_transformer_hocba.pth
  checkpoint: ./hocba_vietocr_fit/checkpoints/vgg19_transformer_hocba_checkpoint.pth
  log: ./hocba_vietocr_fit/logs/train.log
  metrics: 1027
  label_smoothing: 0.1
  early_stopping:
    enabled: true
    patience: 8
    min_delta: 0.0005

dataset:
  name: hocba_cells
  data_root: ./hocba_vietocr_fit/ocr_data
  train_annotation: annotation_train.txt
  valid_annotation: annotation_val.txt
  image_height: 32
  image_min_width: 32
  image_max_width: 768

dataloader:
  num_workers: 2
  pin_memory: true

aug:
  image_aug: true
  masked_language_model: true

predictor:
  beamsearch: false

quiet: false

wandb:
  enabled: false
  project: hocba-vietocr
  name: vgg19-transformer-hocba
'''
config = yaml.safe_load(BASE_CONFIG_YAML)

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
config['device'] = device
config['dataset']['data_root'] = str(OCR_DATA)
config['dataset']['train_annotation'] = TRAIN_ANNOTATION
config['dataset']['valid_annotation'] = VALID_ANNOTATION
config['trainer']['export'] = str(WORK_ROOT / 'weights' / 'vgg19_transformer_hocba.pth')
config['trainer']['checkpoint'] = str(WORK_ROOT / 'checkpoints' / 'vgg19_transformer_hocba_checkpoint.pth')
config['trainer']['log'] = str(WORK_ROOT / 'logs' / 'train.log')

config['optimizer']['max_lr'] = 1e-4
config['optimizer']['pct_start'] = 0.15
config['trainer']['batch_size'] = 32
config['trainer']['iters'] = 6000
config['trainer']['valid_every'] = 250
config['trainer']['print_every'] = 50
config['trainer']['metrics'] = 1027
config['trainer']['label_smoothing'] = 0.1
config['trainer']['early_stopping'] = {'enabled': True, 'patience': 8, 'min_delta': 0.0005}

USE_WANDB = True
WANDB_API_KEY = ''
if USE_WANDB and WANDB_API_KEY:
    import wandb
    wandb.login(key=WANDB_API_KEY)
else:
    USE_WANDB = False
config.setdefault('wandb', {})['enabled'] = bool(USE_WANDB)
config['wandb']['project'] = 'hocba-vietocr'
config['wandb']['name'] = 'vgg19-transformer-hocba-kaggle'

print('device:', device)
print('vocab chars:', len(config['vocab']), 'classes:', len(set(config['vocab'])) + 4)
print('data root:', config['dataset']['data_root'])


## Train Helpers

In [ ]:
def remove_lmdb_dirs(config: Dict[str, Any]) -> None:
    dataset_name = config['dataset']['name']
    for prefix in ('train', 'valid'):
        lmdb_path = Path(f'{prefix}_{dataset_name}')
        if lmdb_path.exists():
            shutil.rmtree(lmdb_path)
            print(f'removed {lmdb_path}')


def patch_vietocr_lmdb_sample_count() -> None:
    import lmdb
    from tqdm import tqdm
    import vietocr.loader.dataloader as dataloader
    import vietocr.tool.create_dataset as create_dataset

    def fixed_create_dataset(outputPath: str, root_dir: str, annotation_path: str) -> None:
        annotation_file = os.path.join(root_dir, annotation_path)
        with open(annotation_file, 'r', encoding='utf-8') as ann_file:
            annotations = [line.rstrip('\n').split('\t', 1) for line in ann_file if line.strip()]
        env = lmdb.open(outputPath, map_size=1099511627776)
        cache = {}
        cnt = 0
        errors = 0
        for image_file, label in tqdm(annotations, ncols=100, desc=f'Create {outputPath}'):
            image_path = os.path.join(root_dir, image_file)
            if not os.path.exists(image_path):
                errors += 1
                continue
            with open(image_path, 'rb') as f:
                image_bin = f.read()
            image_buf = np.frombuffer(image_bin, dtype=np.uint8)
            image = cv2.imdecode(image_buf, cv2.IMREAD_GRAYSCALE)
            if image is None or image.size == 0:
                errors += 1
                continue
            img_h, img_w = image.shape[:2]
            cache[f'image-{cnt:09d}'] = image_bin
            cache[f'label-{cnt:09d}'] = label.encode()
            cache[f'path-{cnt:09d}'] = image_file.encode()
            cache[f'dim-{cnt:09d}'] = np.array([img_h, img_w], dtype=np.int32).tobytes()
            cnt += 1
            if cnt % 1000 == 0:
                create_dataset.writeCache(env, cache)
                cache = {}
        cache['num-samples'] = str(cnt).encode()
        create_dataset.writeCache(env, cache)
        if errors:
            print(f'Remove {errors} invalid images')
        print(f'Created dataset with {cnt} samples')

    create_dataset.createDataset = fixed_create_dataset
    dataloader.createDataset = fixed_create_dataset


def configure_label_smoothing(trainer: Any, config: Dict[str, Any]) -> None:
    from vietocr.optim.labelsmoothingloss import LabelSmoothingLoss
    smoothing = float(config['trainer'].get('label_smoothing', 0.1))
    trainer.criterion = LabelSmoothingLoss(len(trainer.vocab), padding_idx=trainer.vocab.pad, smoothing=smoothing)
    print(f'label_smoothing={smoothing}')


def setup_wandb(config: Dict[str, Any]) -> Optional[Any]:
    wandb_config = config.get('wandb', {})
    if not wandb_config.get('enabled', False):
        return None
    import wandb
    return wandb.init(project=wandb_config.get('project', 'hocba-vietocr'), name=wandb_config.get('name'), config=config)


def log_line(trainer: Any, message: str) -> None:
    print(message)
    if hasattr(trainer, 'logger'):
        trainer.logger.log(message)


def train_with_callbacks(trainer: Any, config: Dict[str, Any], wandb_run: Optional[Any]) -> None:
    total_loss = 0.0
    total_loader_time = 0.0
    total_gpu_time = 0.0
    best_acc = -1.0
    validations_without_improvement = 0
    early_config = config['trainer'].get('early_stopping', {})
    early_enabled = bool(early_config.get('enabled', True))
    patience = int(early_config.get('patience', 8))
    min_delta = float(early_config.get('min_delta', 0.0005))
    data_iter = iter(trainer.train_gen)
    for _ in range(trainer.num_iters):
        trainer.iter += 1
        start = time.time()
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(trainer.train_gen)
            batch = next(data_iter)
        total_loader_time += time.time() - start
        start = time.time()
        loss = trainer.step(batch)
        total_gpu_time += time.time() - start
        total_loss += loss
        trainer.train_losses.append((trainer.iter, loss))
        if trainer.iter % trainer.print_every == 0:
            avg_loss = total_loss / trainer.print_every
            lr = trainer.optimizer.param_groups[0]['lr']
            message = f'iter: {trainer.iter:06d} - train loss: {avg_loss:.3f} - lr: {lr:.2e} - load time: {total_loader_time:.2f} - gpu time: {total_gpu_time:.2f}'
            log_line(trainer, message)
            if wandb_run:
                wandb_run.log({'train/loss': avg_loss, 'train/lr': lr, 'time/load': total_loader_time, 'time/gpu': total_gpu_time}, step=trainer.iter)
            total_loss = 0.0
            total_loader_time = 0.0
            total_gpu_time = 0.0
        if trainer.valid_annotation and trainer.iter % trainer.valid_every == 0:
            val_loss = trainer.validate()
            acc_full_seq, acc_per_char = trainer.precision(trainer.metrics)
            message = f'iter: {trainer.iter:06d} - valid loss: {val_loss:.3f} - acc full seq: {acc_full_seq:.4f} - acc per char: {acc_per_char:.4f}'
            log_line(trainer, message)
            if wandb_run:
                wandb_run.log({'valid/loss': val_loss, 'valid/acc_full_seq': acc_full_seq, 'valid/acc_per_char': acc_per_char, 'valid/best_acc_full_seq': max(best_acc, acc_full_seq)}, step=trainer.iter)
            if acc_full_seq > best_acc + min_delta:
                trainer.save_weights(trainer.export_weights)
                trainer.save_checkpoint(trainer.checkpoint)
                best_acc = acc_full_seq
                validations_without_improvement = 0
                log_line(trainer, f'new best full-sequence accuracy: {best_acc:.4f}')
            else:
                validations_without_improvement += 1
                log_line(trainer, f'early stopping wait: {validations_without_improvement}/{patience} (best acc full seq: {best_acc:.4f})')
            if early_enabled and validations_without_improvement >= patience:
                log_line(trainer, f'early stopping at iter {trainer.iter:06d}')
                break
    if wandb_run:
        wandb_run.finish()


## Train Directly In Notebook

In [ ]:
import torch
import torchvision
print('torch/torchvision:', torch.__version__, torchvision.__version__)

from vietocr.model.trainer import Trainer
from vietocr.tool.config import Cfg

patch_vietocr_lmdb_sample_count()
remove_lmdb_dirs(config)
trainer = Trainer(Cfg(config), pretrained=True)
configure_label_smoothing(trainer, config)
wandb_run = setup_wandb(config)
train_with_callbacks(trainer, config, wandb_run)


## Check Output

In [ ]:
WEIGHTS = Path(config['trainer']['export'])
LOG = Path(config['trainer']['log'])
print('weights exists:', WEIGHTS.exists(), WEIGHTS)
print('log exists:', LOG.exists(), LOG)
if LOG.exists():
    lines = LOG.read_text(encoding='utf-8', errors='replace').splitlines()
    print('\n'.join(lines[-80:]))


In [ ]:
ARTIFACT_ZIP = Path('/kaggle/working/hocba_vietocr_artifacts.zip') if Path('/kaggle').exists() else WORK_ROOT.parent / 'hocba_vietocr_artifacts.zip'
!cd {WORK_ROOT.parent} && zip -qr {ARTIFACT_ZIP} hocba_vietocr_run/weights hocba_vietocr_run/checkpoints hocba_vietocr_run/logs hocba_vietocr_run/quality_audit
print('artifact:', ARTIFACT_ZIP)
